# Explore: beyond n mod 6

**Open this after session 2.** It gives away how session 2 ends.

Each experiment below retrains the session 1 network with one or two settings changed. A run takes about a minute.

The training picture here shows more than it did in session 1. Along with "exactly right", it draws a curve for each piece that the Chinese Remainder Theorem splits the modulus into, plus dashed curves for the training numbers.

## How to use this notebook

- Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
- Cells marked ✏️ need you to type something before you run them. The comments in the cell tell you what.
- If you get stuck, click **Show answer** under the cell.
- If a cell shows an error, look for the yellow box under it. It explains what probably went wrong.

In [ ]:
# Setup: find the course files. On Colab, download them, or update them if already downloaded.
import os, sys, shutil
REPO = "https://github.com/icarm/SACNAS-2026"
if "google.colab" in sys.modules:
    if os.path.exists("/content/course/mlmath.py"):
        os.system("git -C /content/course pull -q --ff-only")  # if this fails, keep the copy we have
    else:
        shutil.rmtree("/content/course", ignore_errors=True)
        if os.system(f"git clone -q --depth 1 {REPO} /content/course") != 0:
            raise RuntimeError("Could not download the course files. Check the wifi, then run this cell again.")
for folder in ["/content/course", ".", ".."]:
    if os.path.exists(os.path.join(folder, "mlmath.py")):
        os.chdir(folder)
        break
else:
    raise RuntimeError("Could not find the course files. Open this notebook from the ml4math-mini-course folder.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for name in ("mlmath", "lab"):
    sys.modules.pop(name, None)  # so the next import loads the newest version
print("Ready.")

In [ ]:
import lab
mlmath = lab.mlmath
print("Ready to experiment.")

## 1. Use the theorem to predict: n mod 12

Since 12 = 4 × 3, and 4 and 3 have no common factor, the theorem says that knowing n mod 12 is the same as knowing n mod 4 and n mod 3. Because 100 is divisible by 4, n mod 4 only depends on the last two digits, while n mod 3 needs every digit.

✏️ Before you run it, predict: how many drops will the loss take, and in what order will the pieces show up? Then type 12 in place of the `...` and run the cell.

In [ ]:
result = lab.run(modulus=...)    # ✏️ type 12

<details><summary>Show answer</summary>

```python
result = lab.run(modulus=12)
```

</details>

## 2. Pick your own modulus

✏️ Choose a modulus. Before you run it, factor it and guess: will it be fast, slow, or stuck? A few to try:

- 10 = 2 × 5. Both pieces only need the last digit.
- 9 = 3 × 3. The theorem can't split this one, because 3 and 3 share a factor.
- 7 is prime, and 10 mod 7 = 3, so there's no digit-sum shortcut.

In [ ]:
result = lab.run(modulus=...)    # ✏️ your modulus

<details><summary>Show answer</summary>

```python
result = lab.run(modulus=10)
```

</details>

## 3. Is 99% the same as knowing the rule?

In this run, no training number has a 7 in the hundreds place. After training, we test the network on exactly those numbers. A mathematician who knows the digit-sum rule would apply it to any digit in any place. Will the network?

✏️ Guessing would be right 17% of the time. Predict how often the network will be right on the hidden numbers. Will it at least get parity? Then type 7 in place of the `...` and run the cell.

In [ ]:
result = lab.run(hide_place="hundreds", hide_digit=...)    # ✏️ type 7

<details><summary>Show answer</summary>

```python
result = lab.run(hide_place="hundreds", hide_digit=7)
```

</details>

✏️ 4712 has a 7 in the hundreds place, and 4012 doesn't. Ask the network about both, then add some numbers of your own.

In [ ]:
# ✏️ Add more numbers, some with a 7 in the hundreds place and some without.
mlmath.ask({"this network": result.model}, [4712, 4012])

<details><summary>Show answer</summary>

```python
mlmath.ask({"this network": result.model}, [4712, 4012, 1700, 1000])
```

</details>

**Discuss.** On ordinary test numbers, this network is right 99% of the time. On the hidden numbers it does far worse than guessing, but it still gets parity right. Why?

The switch for "7 in the hundreds place" was never on during training, so its connections never moved from their random starting values. The network never learned "add up the digits". Instead it learned a separate table of digit groups for each place, and that table has no entry for a digit it never saw. So a 99% score on numbers like the training numbers is good evidence, but it isn't proof that the network knows the rule.

## 4. Change anything

✏️ Change one setting at a time, make a prediction, and run. Here are some ideas:

| Try | Question |
| --- | --- |
| `lab.run(width=16)` | Session 1 used 512 hidden units. With only 16, does it still find the mod 3 rule? |
| `lab.run(train_frac=0.1)` | Train on 10% of the numbers instead of half. Which rule survives, and which turns into memorizing? |
| `lab.run(optimizer="Adam", epochs=200)` | Adam is the training method most people use. What happens to the flat stretch? |
| `lab.run(encoding="one-hot base 6 digits")` | Write every number in base 6. Now n mod 6 is just the last digit. What happens to the two drops? |
| `lab.run(optimizer="Adam", weight_decay=3.0, epochs=200)` | Weight decay penalizes the network for large connection strengths. Can it still afford the mod 3 rule? |

Run `lab.menu()` to see every setting you can change.

In [ ]:
# ✏️ Change the setting, then run.
result = lab.run(width=16)

To look inside the network from your last run, use the swap table from session 2. Change `"thousands"` to look at a different place.

In [ ]:
mlmath.plot_swap_tables({"this network": result.model}, result.data, "thousands")

## Buttons (optional)

If you'd rather click than type, this panel runs the same experiments, and a few more, with buttons.

In [ ]:
lab.panel()

## Going further: your own encoding

An encoding is any function that takes `(numbers, n_digits)` and returns one row of inputs for each number. `encode_base6` in `lab.py` is a short example you can copy. To try yours, register it and run it:

```python
lab.ENCODINGS["my encoding"] = my_encoding
result = lab.run(encoding="my encoding")
```